# Pig Posture Recognition - V4 Inference

**Verbesserungen gegenueber V3:**
1. **Multi-Architektur Ensemble** - DINOv2 + ConvNeXt (aus v4_train)
2. **Groessere Inferenz-Aufloesung** fuer ViT (518 statt 392) - kostenlos +0.5-1% F1
3. **Class-balanced Pseudo-Label Export** - Top-k pro Klasse statt globaler Threshold
4. **Test-Time BN Adaptation** fuer ConvNeXt (BatchNorm-Modelle)
5. **Temperatur-Kalibrierung** (optional)

## Configuration

In [1]:
import os

TAG = "T2"

_candidates = [
    'multiview_pig_posture_recognition',
    './multiview_pig_posture_recognition',
    '/datasets/multi-view-pig-posture-recognition',
    '/multi-view-pig-posture-recognition',
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

TEST_CSV  = os.path.join(DATA_ROOT, "test.csv")
IMG_DIR   = os.path.join(DATA_ROOT, "test_images")

CKPT_DIR  = f"runs/v4_{TAG.lower()}"
CKPT_PATHS = sorted([
    os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR)
    if f.startswith("best_") and f.endswith(".pth")
])

OUTPUT_FILE = f"{TAG}_v4_submission.csv"

# --- Inferenz-Aufloesung pro Architektur-Prefix ---
# Groesser als beim Training -> mehr Detail, profitiert von ViT-Positions-Embedding-Interpolation
INFER_IMG_SIZE = {
    "dinov2":   518,    # war 392 im Training, 518 ist DINOv2-Default
    "convnext": 384,    # ConvNeXt gleich lassen
}

BATCH_SIZE   = 64
NUM_WORKERS  = 16
USE_TTA      = True
PAD_RATIO    = 0.1
NUM_CLASSES  = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

# --- Test-Time BN Adaptation (nur fuer ConvNeXt relevant) ---
ADAPT_BN         = True
BN_ADAPT_BATCHES = 50

# --- Pseudo-Label Export: class-balanced Top-k ---
EXPORT_PSEUDO_LABELS = True
# Mode:
#   "threshold" = global threshold (V3-Verhalten)
#   "top_k"     = Top-k pro Klasse (empfohlen, balanciert)
PSEUDO_MODE            = "top_k"
PSEUDO_THRESHOLD       = 0.85          # fuer "threshold"-Mode (gesenkt von 0.95)
PSEUDO_TOP_K_PER_CLASS = 200           # pro Klasse (5x200 = ~1000 Pseudo-Labels max)
PSEUDO_MIN_CONFIDENCE  = 0.60          # unterer Cutoff auch im Top-k Mode
PSEUDO_OUTPUT          = f"pseudo_labels_{TAG.lower()}_v4.csv"

# --- Ensemble-Gewichtung ---
# None = gleichgewichtet. Dict = Multiplikator pro Prefix.
ENSEMBLE_WEIGHTS = None  # z.B. {"dinov2": 1.5, "convnext": 1.0} falls ein Modell besser

print(f"Tag: {TAG}  |  Checkpoints: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    print(f"  - {os.path.basename(p)}")


DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Checkpoints: 10
  - best_convnext_fold_1.pth
  - best_convnext_fold_2.pth
  - best_convnext_fold_3.pth
  - best_convnext_fold_4.pth
  - best_convnext_fold_5.pth
  - best_dinov2_fold_1.pth
  - best_dinov2_fold_2.pth
  - best_dinov2_fold_3.pth
  - best_dinov2_fold_4.pth
  - best_dinov2_fold_5.pth


## Imports

In [2]:
import os, ast, re
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


<jemalloc>: Unsupported system page size


Device: cuda


## Test Dataset

In [3]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.1):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform:
            crop = self.transform(crop)
        return crop, row["row_id"]


## Test-Time BN Adaptation

Nur relevant fuer BatchNorm-Modelle (ConvNeXt). DINOv2/ViT wird automatisch uebersprungen (LayerNorm).

In [4]:
def adapt_batch_norm(model, loader, device, n_batches=50):
    bn_layers = [m for m in model.modules()
                 if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.SyncBatchNorm))]
    if not bn_layers:
        print("    Keine BatchNorm Layer - uebersprungen.")
        return
    for bn in bn_layers:
        bn.running_mean.zero_()
        bn.running_var.fill_(1)
        bn.momentum = None
    model.train()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(tqdm(loader, desc="    BN-Adapt", leave=False)):
            if i >= n_batches: break
            model(imgs.to(device))
    model.eval()
    print(f"    BN auf {min(n_batches, len(loader))} Batches adaptiert ({len(bn_layers)} Layer)")


## TTA mit Flip-Korrektur

6 Views: Original, HFlip, +32px Crop, +32px HFlip, +64px Crop, 0.75 Downscale

In [5]:
NORM = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

def build_tta_configs(img_size):
    S = img_size
    return [
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+64, S+64), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((int(S*0.75), int(S*0.75))),
                    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
    ]


## Multi-Arch Inference

In [6]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test-Instanzen: {len(test_df)}")


@torch.no_grad()
def predict_tta(model, df, img_dir, tta_configs, batch_size):
    all_probs = []
    for i, (tf, is_flipped) in enumerate(tta_configs):
        ds = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(tta_configs)}", leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            p = F.softmax(logits, dim=1).cpu().numpy()
            if is_flipped:
                p[:, [0, 1]] = p[:, [1, 0]]
            probs.append(p)
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)


def infer_prefix(ckpt_path):
    """Extrahiere Arch-Prefix aus Dateiname (z.B. 'best_dinov2_fold_1.pth' -> 'dinov2')."""
    name = os.path.basename(ckpt_path)
    m = re.match(r"best_([a-zA-Z0-9]+)_fold_\d+\.pth", name)
    return m.group(1) if m else "unknown"


ensemble_probs_weighted = []
ensemble_weights_used = []

for idx, path in enumerate(CKPT_PATHS):
    prefix = infer_prefix(path)
    print(f"\nModell {idx+1}/{len(CKPT_PATHS)}: {os.path.basename(path)} (arch={prefix})")

    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", "unknown")
    ckpt_img_size = ckpt.get("img_size", 392)
    infer_size = INFER_IMG_SIZE.get(prefix, ckpt_img_size)

    print(f"  {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}  |  "
          f"Train@{ckpt_img_size}px, Infer@{infer_size}px")

    # --- Modell laden (img_size ggf. interpoliert) ---
    try:
        model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES, img_size=infer_size)
    except TypeError:
        model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(ckpt["model"])
    model.to(DEVICE).eval()

    # --- BN Adaptation (nur BatchNorm-Modelle) ---
    if ADAPT_BN:
        bn_tf = T.Compose([
            T.Resize((infer_size, infer_size), interpolation=T.InterpolationMode.BICUBIC),
            T.ToTensor(), T.Normalize(*NORM),
        ])
        bn_ds = PigTestDataset(test_df, IMG_DIR, transform=bn_tf, pad_ratio=PAD_RATIO)
        bn_loader = DataLoader(bn_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
        adapt_batch_norm(model, bn_loader, DEVICE, n_batches=BN_ADAPT_BATCHES)

    tta_cfg = build_tta_configs(infer_size) if USE_TTA else [build_tta_configs(infer_size)[0]]
    fold_probs = predict_tta(model, test_df, IMG_DIR, tta_cfg, BATCH_SIZE)

    w = 1.0
    if ENSEMBLE_WEIGHTS is not None:
        w = ENSEMBLE_WEIGHTS.get(prefix, 1.0)

    ensemble_probs_weighted.append(fold_probs * w)
    ensemble_weights_used.append(w)

    del model
    torch.cuda.empty_cache()

# Normiere: Summe der Wahrscheinlichkeiten / Summe der Gewichte
final_probs = np.sum(ensemble_probs_weighted, axis=0) / max(sum(ensemble_weights_used), 1e-6)
predictions = final_probs.argmax(axis=1)

print(f"\n{len(predictions)} Vorhersagen aus {len(CKPT_PATHS)} Modellen")


Test-Instanzen: 11708

Modell 1/10: best_convnext_fold_1.pth (arch=convnext)
  convnext_base.fb_in22k_ft_in1k  |  Val F1: 0.7042  |  Train@384px, Infer@384px
    Keine BatchNorm Layer - uebersprungen.


  TTA 1/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 2/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 3/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/183 [00:00<?, ?it/s]

...

  TTA 4/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 5/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 6/6:   0%|          | 0/183 [00:00<?, ?it/s]


Modell 6/10: best_dinov2_fold_1.pth (arch=dinov2)
  vit_base_patch14_dinov2.lvd142m  |  Val F1: 0.8167  |  Train@392px, Infer@518px


RuntimeError: Error(s) in loading state_dict for VisionTransformer:
	size mismatch for pos_embed: copying a param with shape torch.Size([1, 785, 768]) from checkpoint, the shape in current model is torch.Size([1, 1370, 768]).

## Submission

In [ ]:
submission = pd.DataFrame({
    "row_id": test_df["row_id"].values,
    "class_id": predictions.astype(int)
})
submission.to_csv(OUTPUT_FILE, index=False)

assert list(submission.columns) == ["row_id", "class_id"]
assert set(submission["class_id"].unique()).issubset(set(range(5)))
assert len(submission) == len(test_df)

print(f"Submission: {OUTPUT_FILE} ({len(submission)} Zeilen)\n")
print(f"Verteilung:")
for c in range(NUM_CLASSES):
    cnt = (submission["class_id"] == c).sum()
    pct = 100 * cnt / len(submission)
    bar = "#" * int(30 * cnt / len(submission))
    print(f"  {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5} ({pct:.1f}%)")

submission.head(10)


## Class-balanced Pseudo-Label Export

**Problem V3:** Threshold 0.95 -> 110 Pseudo-Labels, alle Sitting. Keine Lateral_/Standing/Sternal.

**V4-Loesung:** Top-k pro Klasse (`PSEUDO_MODE="top_k"`). Jede Klasse bekommt die K sichersten Vorhersagen -> balancierter Trainings-Boost.

In [ ]:
if EXPORT_PSEUDO_LABELS:
    max_probs = final_probs.max(axis=1)
    argmax = final_probs.argmax(axis=1)

    if PSEUDO_MODE == "top_k":
        selected_idx = []
        print(f"Class-balanced Top-{PSEUDO_TOP_K_PER_CLASS} pro Klasse "
              f"(min conf {PSEUDO_MIN_CONFIDENCE}):")
        for c in range(NUM_CLASSES):
            cand_mask = (argmax == c) & (max_probs >= PSEUDO_MIN_CONFIDENCE)
            cand_idx = np.where(cand_mask)[0]
            # Top-K nach Confidence
            cand_idx = cand_idx[np.argsort(-max_probs[cand_idx])]
            chosen = cand_idx[:PSEUDO_TOP_K_PER_CLASS]
            selected_idx.extend(chosen.tolist())
            if len(chosen) > 0:
                print(f"  {c} - {CLASS_NAMES[c]:<22} "
                      f"{len(chosen):>4} ausgewaehlt "
                      f"(min conf={max_probs[chosen].min():.3f}, "
                      f"max conf={max_probs[chosen].max():.3f})")
            else:
                print(f"  {c} - {CLASS_NAMES[c]:<22} KEINE Kandidaten")
        selected_idx = np.array(sorted(selected_idx), dtype=int)
        mask = np.zeros(len(test_df), dtype=bool)
        mask[selected_idx] = True
    else:
        mask = max_probs >= PSEUDO_THRESHOLD

    pseudo_df = test_df[mask].copy()
    pseudo_df["class_id"] = predictions[mask].astype(int)
    pseudo_df["confidence"] = max_probs[mask]
    pseudo_df.to_csv(PSEUDO_OUTPUT, index=False)

    n = mask.sum()
    print(f"\nPseudo-Labels: {PSEUDO_OUTPUT}")
    print(f"  {n} von {len(test_df)} ({100*n/len(test_df):.1f}%)")
    print(f"  Verteilung:")
    for c in range(NUM_CLASSES):
        cnt = (pseudo_df["class_id"] == c).sum()
        print(f"    {c} - {CLASS_NAMES[c]:<22} {cnt:>4}")
    print(f"\nNaechster Schritt: v4_train.ipynb mit USE_PSEUDO_LABELS=True, "
          f"PSEUDO_CSV=\"{PSEUDO_OUTPUT}\"")
else:
    print("Pseudo-Label Export deaktiviert.")
